# #LexisNexis Fraud Detection - Task 2: Address Class Imbalance

Compares class weighting vs. SMOTE using a logistic regression baseline. Setup (data, features, preprocessing) is copied from Task 1. All preprocessing is fit on the 1994 training set only; SMOTE is applied to training data only.


In [ ]:
!pip install -q imbalanced-learn
!git clone -q https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims


/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("data/fraud_oracle_cleaned.csv")
TARGET = "FraudFound_P"

SAFE_FEATURES = [
    "Month", "WeekOfMonth", "DayOfWeek", "MonthClaimed", "WeekOfMonthClaimed",
    "DayOfWeekClaimed", "Make", "VehicleCategory", "VehiclePrice", "AgeOfVehicle",
    "NumberOfCars", "AccidentArea", "Fault", "Sex", "MaritalStatus", "Age",
    "Age_missing", "AgeOfPolicyHolder", "PolicyType", "BasePolicy", "Deductible",
    "DriverRating", "AgentType", "Days_Policy_Accident", "Days_Policy_Claim",
    "PastNumberOfClaims",
]

train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
test  = df[df["Year"] == 1996].copy()

MONTH_NUM = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
             "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}

VEHICLE_AGE_MAP = {"new": 0, "2 years": 2, "3 years": 3, "4 years": 4,
                   "5 years": 5, "6 years": 6, "7 years": 7, "more than 7": 8}

def add_engineered_features(data):
    data = data.copy()
    data["AgeBand"] = pd.cut(
        data["Age"], bins=[0, 24, 34, 44, 54, 64, np.inf],
        labels=["<25", "25-34", "35-44", "45-54", "55-64", "65+"])
    data["AgeBand"] = data["AgeBand"].astype("object").fillna("Missing")
    data["VehicleAgeApprox"] = data["AgeOfVehicle"].map(VEHICLE_AGE_MAP)
    accident_month = data["Month"].map(MONTH_NUM)
    claimed_month = data["MonthClaimed"].map(MONTH_NUM)
    data["ClaimLagMonths"] = (claimed_month - accident_month) % 12
    data["WeekendAccident"] = data["DayOfWeek"].isin(["Saturday", "Sunday"]).astype(int)
    data["WeekendClaim"] = data["DayOfWeekClaimed"].isin(["Saturday", "Sunday"]).astype(int)
    return data

X_train = add_engineered_features(train[SAFE_FEATURES])
X_val   = add_engineered_features(val[SAFE_FEATURES])
X_test  = add_engineered_features(test[SAFE_FEATURES])
y_train, y_val, y_test = train[TARGET], val[TARGET], test[TARGET]

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()
grouped_categories = ["Make", "PolicyType"]
regular_categories = [c for c in categorical_features if c not in grouped_categories]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
grouped_category_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=0.01)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, regular_categories),
    ("grouped_categorical", grouped_category_pipeline, grouped_categories),
])

preprocessor.fit(X_train)  # fit on training set only
X_train_processed = preprocessor.transform(X_train)
X_val_processed   = preprocessor.transform(X_val)
X_test_processed  = preprocessor.transform(X_test)

print("Processed shapes:", X_train_processed.shape, X_val_processed.shape, X_test_processed.shape)

Processed shapes: (6141, 128) (5195, 128) (4083, 128)


Task 2: Class weighting vs. SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

print("Train fraud rate:", round(y_train.mean(), 4))
print(y_train.value_counts())


Train fraud rate: 0.0666
FraudFound_P
0    5732
1     409
Name: count, dtype: int64


In [ ]:
def evaluate(name, model, X_tr, y_tr, X_ev, y_ev, top_fracs=(0.10, 0.20)):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]
    pred = (proba >= 0.5).astype(int)
    row = {
        "Approach": name,
        "Precision": precision_score(y_ev, pred, zero_division=0),
        "Recall": recall_score(y_ev, pred),
        "F1": f1_score(y_ev, pred),
        "PR-AUC": average_precision_score(y_ev, proba),
    }
    # % of fraud captured in the top 10% / 20% highest-scored claims (for Task 6)
    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    for f in top_fracs:
        k = int(len(y_sorted) * f)
        row[f"Fraud captured top {int(f*100)}%"] = y_sorted[:k].sum() / y_sorted.sum()
    return row

lr_kwargs = dict(max_iter=1000, random_state=42)
results = []

# 1) Reference: no imbalance handling
results.append(evaluate("LR baseline (no handling)", LogisticRegression(**lr_kwargs),
                        X_train_processed, y_train, X_val_processed, y_val))

# 2) Class weighting
results.append(evaluate("LR + class_weight='balanced'",
                        LogisticRegression(class_weight="balanced", **lr_kwargs),
                        X_train_processed, y_train, X_val_processed, y_val))

# 3) SMOTE on training data only
X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train_processed, y_train)
print("After SMOTE:", X_train_sm.shape, y_train_sm.value_counts().to_dict())
results.append(evaluate("LR + SMOTE", LogisticRegression(**lr_kwargs),
                        X_train_sm, y_train_sm, X_val_processed, y_val))


After SMOTE: (11464, 128) {0: 5732, 1: 5732}


In [ ]:
results_df = pd.DataFrame(results).set_index("Approach").round(4)
results_df

,Precision,Recall,F1,PR-AUC,Fraud captured top 10%,Fraud captured top 20%
Approach,,,,,,
LR baseline (no handling),0.0556,0.0033,0.0063,0.0983,0.1628,0.3488
LR + class_weight='balanced',0.1007,0.4319,0.1633,0.0946,0.1927,0.3355
LR + SMOTE,0.1020,0.4020,0.1627,0.0950,0.1860,0.3322


Chosen strategy for Task 3 (PR-AUC first; recall breaks ties)

In [ ]:
candidates = results_df.drop(index="LR baseline (no handling)")
best = candidates.sort_values(["PR-AUC", "Recall"], ascending=False).index[0]
print("Chosen imbalance strategy:", best)

Chosen imbalance strategy: LR + SMOTE


Fraud is only about 7% of claims, so we compared class weighting and SMOTE (training set only) using logistic regression, evaluated on the 1995 validation set.

The baseline almost never caught fraud (recall 0.003, F1 0.006). Class weighting raised recall to 0.432 and F1 to 0.163, while SMOTE reached 0.402 and 0.163. Precision rose from 0.056 to about 0.10 for both. PR-AUC stayed flat (about 0.095-0.098), so neither method improved how well the model ranks risky claims.

We selected class weighting (TBD) for its simplicity and slightly better recall, F1, and top 10-20% fraud capture. It will be used in Tasks 3 and 4.